# MITRE ATT&CK Knowledge Graph — v2.0
**Full pipeline:** Tactics → Techniques → Sub-techniques → Groups → Software → CAPEC → CWE → CVE

**CVE Source:** CISA KEV, NVD


## Graph schema
```
(CVE)-[:CAUSE_WEAKNESS]->(CWE)
(CWE)<-[:TARGETS_WEAKNESS]-(CAPEC)
(CAPEC)-[:MAPS_TO_TECHNIQUE]->(Technique)
(Technique)-[:BELONGS_TO]->(Tactic)
(Technique)-[:HAS_SUBTECHNIQUE]->(Technique)   # sub-techniques
(Group)-[:USES]->(Technique)
(Group)-[:USES]->(Software)
(Software)-[:USES]->(Technique)
(CAPEC)-[:CHILD_OF]->(CAPEC)                   # CAPEC hierarchy
```

## Run order
1. Cell 1 – Constraints & indexes (run once)
2. Cell 2 – Ingest ATT&CK (Tactics + Techniques + Sub-techniques + Groups + Software)
3. Cell 3 – Ingest CAPEC nodes + hierarchy + CWE links + ATT&CK links
4. Cell 4 – Ingest CVEs (NVD) with CWE bridge
5. Cell 5 – Verify graph health


---
## Cell 1 — Config, driver, constraints

In [1]:
from neo4j import GraphDatabase

URI  = "bolt://localhost:7687"
AUTH = ("neo4j", "kg_mitre_v1.1")   # ← change if needed

driver = GraphDatabase.driver(URI, auth=AUTH)

CONSTRAINTS = [
    # Node uniqueness — prevents duplicates on re-runs
    "CREATE CONSTRAINT IF NOT EXISTS FOR (t:Technique)  REQUIRE t.stix_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (ta:Tactic)    REQUIRE ta.stix_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (g:Group)      REQUIRE g.stix_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (s:Software)   REQUIRE s.stix_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (c:CAPEC)      REQUIRE c.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (w:CWE)        REQUIRE w.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (v:CVE)        REQUIRE v.id IS UNIQUE",
    # Indexes for fast lookups
    "CREATE INDEX IF NOT EXISTS FOR (t:Technique)  ON (t.mitre_id)",
    "CREATE INDEX IF NOT EXISTS FOR (ta:Tactic)    ON (ta.mitre_id)",
]

with driver.session() as s:
    for stmt in CONSTRAINTS:
        s.run(stmt)

print("✅ Constraints and indexes ready.")

✅ Constraints and indexes ready.


---
## Cell 2 — Ingest ATT&CK STIX bundle
Ingests: **Tactics**, **Techniques**, **Sub-techniques**, **Groups**, **Software (malware + tools)**  
Relationships: `BELONGS_TO`, `HAS_SUBTECHNIQUE`, `USES`

In [2]:
import json
from stix2 import MemoryStore, Filter

ATTACK_FILE = 'enterprise-attack.json' 

def _get(obj, key, default=None):
    """Unified getter: works for both stix2 objects and plain dicts."""
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)

def _id(obj):
    """Get the STIX id field from either a stix2 object or a dict."""
    return obj['id'] if isinstance(obj, dict) else obj.id

def _name(obj):
    return obj['name'] if isinstance(obj, dict) else obj.name

def mitre_id(obj):
    """Extract the MITRE ATT&CK external ID (e.g. T1059, TA0002, G0016)."""
    refs = _get(obj, 'external_references', [])
    for ref in refs:
        src = ref.get('source_name') if isinstance(ref, dict) else getattr(ref, 'source_name', '')
        if src == 'mitre-attack':
            return ref.get('external_id') if isinstance(ref, dict) else getattr(ref, 'external_id', None)
    return _id(obj)  # fallback to STIX id

def url(obj):
    refs = _get(obj, 'external_references', [])
    for ref in refs:
        src = ref.get('source_name') if isinstance(ref, dict) else getattr(ref, 'source_name', '')
        if src == 'mitre-attack':
            return ref.get('url', '') if isinstance(ref, dict) else getattr(ref, 'url', '')
    return ''

def run_full_attack_ingestion(file_path):
    print("Loading STIX bundle…")
    with open(file_path) as f:
        bundle = json.load(f)
    store = MemoryStore(stix_data=bundle)

    with driver.session() as s:
        # ── 1. TACTICS (x-mitre-tactic) ──────────────────────────────────────
        print("Ingesting Tactics…")
        tactics = store.query([Filter('type', '=', 'x-mitre-tactic')])
        for t in tactics:
            mid = mitre_id(t)
            s.run("""
                MERGE (ta:Tactic {stix_id: $stix_id})
                SET ta.mitre_id   = $mid,
                    ta.name       = $name,
                    ta.short_name = $short,
                    ta.description= $desc,
                    ta.url        = $url
            """, stix_id=_id(t), mid=mid, name=_name(t),
                 short=_get(t, 'x_mitre_shortname', ''),
                 desc=_get(t, 'description', ''), url=url(t))
        print(f"  → {len(tactics)} tactics")

        # ── 2. TECHNIQUES + SUB-TECHNIQUES (attack-pattern) ──────────────────
        print("Ingesting Techniques & Sub-techniques…")
        techniques = store.query([Filter('type', '=', 'attack-pattern')])
        # Filter out revoked/deprecated
        techniques = [t for t in techniques
                      if not t.get('revoked') and not t.get('x_mitre_deprecated')]
        for t in techniques:
            mid = mitre_id(t)
            is_sub = '.' in mid
            platforms = t.get('x_mitre_platforms', [])
            s.run("""
                MERGE (tech:Technique {stix_id: $stix_id})
                SET tech.mitre_id    = $mid,
                    tech.name        = $name,
                    tech.description = $desc,
                    tech.is_subtechnique = $is_sub,
                    tech.platforms   = $platforms,
                    tech.url         = $url
            """, stix_id=t.id, mid=mid, name=t.name,
                 desc=t.get('description', ''), is_sub=is_sub,
                 platforms=platforms, url=url(t))
        print(f"  → {len(techniques)} techniques/sub-techniques")

        # ── 3. GROUPS (intrusion-set) ─────────────────────────────────────────
        print("Ingesting Groups (Threat Actors)…")
        groups = store.query([Filter('type', '=', 'intrusion-set')])
        groups = [g for g in groups
                  if not g.get('revoked') and not g.get('x_mitre_deprecated')]
        for g in groups:
            mid = mitre_id(g)
            aliases = g.get('aliases', [])
            s.run("""
                MERGE (gr:Group {stix_id: $stix_id})
                SET gr.mitre_id   = $mid,
                    gr.name       = $name,
                    gr.aliases    = $aliases,
                    gr.description= $desc,
                    gr.url        = $url
            """, stix_id=g.id, mid=mid, name=g.name,
                 aliases=aliases, desc=g.get('description', ''), url=url(g))
        print(f"  → {len(groups)} groups")

        # ── 4. SOFTWARE (malware + tool) ──────────────────────────────────────
        print("Ingesting Software (Malware + Tools)…")
        software = (
            store.query([Filter('type', '=', 'malware')]) +
            store.query([Filter('type', '=', 'tool')])
        )
        software = [sw for sw in software
                    if not sw.get('revoked') and not sw.get('x_mitre_deprecated')]
        for sw in software:
            mid = mitre_id(sw)
            s.run("""
                MERGE (so:Software {stix_id: $stix_id})
                SET so.mitre_id   = $mid,
                    so.name       = $name,
                    so.type       = $sw_type,
                    so.description= $desc,
                    so.url        = $url
            """, stix_id=sw.id, mid=mid, name=sw.name,
                 sw_type=sw.type, desc=sw.get('description', ''), url=url(sw))
        print(f"  → {len(software)} software items")

        # ── 5. RELATIONSHIPS ──────────────────────────────────────────────────
        print("Building relationships…")
        rels = store.query([Filter('type', '=', 'relationship')])

        tactic_phase_map = {_get(t, 'x_mitre_shortname'): _id(t) for t in tactics}

        uses_count       = 0
        subtechnique_count = 0
        tactic_count     = 0

        # 5a. Technique → Tactic (via kill_chain_phases on each technique)
        for t in techniques:
            for phase in t.get('kill_chain_phases', []):
                short = phase.get('phase_name')
                ta_stix = tactic_phase_map.get(short)
                if ta_stix:
                    s.run("""
                        MATCH (tech:Technique {stix_id: $tech_id})
                        MATCH (ta:Tactic {stix_id: $ta_id})
                        MERGE (tech)-[:BELONGS_TO]->(ta)
                    """, tech_id=t.id, ta_id=ta_stix)
                    tactic_count += 1

        # 5b. Sub-technique → Parent technique
        for rel in rels:
            if rel.relationship_type == 'subtechnique-of':
                s.run("""
                    MATCH (parent:Technique {stix_id: $parent_id})
                    MATCH (sub:Technique   {stix_id: $sub_id})
                    MERGE (parent)-[:HAS_SUBTECHNIQUE]->(sub)
                """, parent_id=rel.target_ref, sub_id=rel.source_ref)
                subtechnique_count += 1

        # 5c. Group/Software USES Technique/Software
        for rel in rels:
            if rel.relationship_type == 'uses':
                s.run("""
                    MATCH (src {stix_id: $src})
                    MATCH (tgt {stix_id: $tgt})
                    MERGE (src)-[:USES]->(tgt)
                """, src=rel.source_ref, tgt=rel.target_ref)
                uses_count += 1

        print(f"  → {tactic_count} BELONGS_TO (Technique→Tactic)")
        print(f"  → {subtechnique_count} HAS_SUBTECHNIQUE")
        print(f"  → {uses_count} USES")

    print("\n✅ ATT&CK ingestion complete!")

run_full_attack_ingestion(ATTACK_FILE)

Loading STIX bundle…
Ingesting Tactics…
  → 14 tactics
Ingesting Techniques & Sub-techniques…
  → 691 techniques/sub-techniques
Ingesting Groups (Threat Actors)…
  → 172 groups
Ingesting Software (Malware + Tools)…
  → 784 software items
Building relationships…
  → 887 BELONGS_TO (Technique→Tactic)
  → 477 HAS_SUBTECHNIQUE
  → 17270 USES

✅ ATT&CK ingestion complete!


---
## Cell 3 — Ingest CAPEC
Ingests: **CAPEC nodes** with metadata  
Relationships: `CHILD_OF` (CAPEC hierarchy), `TARGETS_WEAKNESS` (CAPEC→CWE), `MAPS_TO_TECHNIQUE` (CAPEC→Technique)

**Fixes vs v1:**
- CAPEC nodes are now created with name + description + likelihood/severity
- CWE link uses `CWE_ID` attribute (not `.text`)
- CAPEC→CAPEC hierarchy is preserved
- Sub-technique IDs (e.g. T1574.010) try exact match first, then parent fallback

In [3]:
import xml.etree.ElementTree as ET

CAPEC_FILE = 'capec_v3.9.xml' 

def ingest_capec_full(file_path):
    tree = ET.parse(file_path)
    root = tree.getroot()
    ns   = {'c': 'http://capec.mitre.org/capec-3'}

    patterns = root.findall('.//c:Attack_Pattern', ns)
    print(f"Found {len(patterns)} CAPEC patterns")

    with driver.session() as s:
        # ── 1. Create all CAPEC nodes ─────────────────────────────────────────
        print("Creating CAPEC nodes…")
        for p in patterns:
            capec_id = f"CAPEC-{p.get('ID')}"
            name     = p.get('Name', '')
            abstr    = p.get('Abstraction', '')
            status   = p.get('Status', '')
            desc_el  = p.find('c:Description', ns)
            desc     = (desc_el.text or '').strip() if desc_el is not None else ''
            likely   = getattr(p.find('c:Likelihood_Of_Attack', ns), 'text', '') or ''
            severity = getattr(p.find('c:Typical_Severity', ns), 'text', '') or ''

            s.run("""
                MERGE (ca:CAPEC {id: $id})
                SET ca.name        = $name,
                    ca.abstraction = $abstr,
                    ca.status      = $status,
                    ca.description = $desc,
                    ca.likelihood  = $likely,
                    ca.severity    = $severity
            """, id=capec_id, name=name, abstr=abstr, status=status,
                 desc=desc, likely=likely, severity=severity)

        # ── 2. CAPEC → CWE  (TARGETS_WEAKNESS) ───────────────────────────────
        print("Linking CAPEC → CWE…")
        cwe_links = 0
        for p in patterns:
            capec_id = f"CAPEC-{p.get('ID')}"
            for rw in p.findall('.//c:Related_Weakness', ns):
                cwe_num = rw.get('CWE_ID')        # e.g. "276"
                if cwe_num:
                    cwe_id = f"CWE-{cwe_num}"
                    s.run("""
                        MATCH (ca:CAPEC {id: $capec_id})
                        MERGE (w:CWE {id: $cwe_id})
                        MERGE (ca)-[:TARGETS_WEAKNESS]->(w)
                    """, capec_id=capec_id, cwe_id=cwe_id)
                    cwe_links += 1
        print(f"  → {cwe_links} TARGETS_WEAKNESS relationships")

        # ── 3. CAPEC hierarchy  (CHILD_OF) ────────────────────────────────────
        print("Building CAPEC hierarchy…")
        hierarchy_links = 0
        for p in patterns:
            child_id = f"CAPEC-{p.get('ID')}"
            for r in p.findall('.//c:Related_Attack_Pattern', ns):
                if r.get('Nature') == 'ChildOf':
                    parent_id = f"CAPEC-{r.get('CAPEC_ID')}"
                    s.run("""
                        MATCH (child:CAPEC  {id: $child_id})
                        MATCH (parent:CAPEC {id: $parent_id})
                        MERGE (child)-[:CHILD_OF]->(parent)
                    """, child_id=child_id, parent_id=parent_id)
                    hierarchy_links += 1
        print(f"  → {hierarchy_links} CHILD_OF relationships")

        # ── 4. CAPEC → ATT&CK Technique  (MAPS_TO_TECHNIQUE) ─────────────────
        print("Linking CAPEC → ATT&CK Techniques…")
        atk_links = 0
        for p in patterns:
            capec_id = f"CAPEC-{p.get('ID')}"
            for m in p.findall('.//c:Taxonomy_Mapping', ns):
                if m.get('Taxonomy_Name') != 'ATTACK':
                    continue
                entry_el = m.find('c:Entry_ID', ns)
                if entry_el is None or not entry_el.text:
                    continue
                raw = entry_el.text.strip()
                tech_id   = raw if raw.startswith('T') else f"T{raw}"  # e.g. T1574.010
                parent_id = tech_id.split('.')[0]                        # e.g. T1574

                result = s.run("""
                    MATCH (ca:CAPEC {id: $capec_id})
                    MATCH (t:Technique)
                    WHERE t.mitre_id = $tech_id OR t.mitre_id = $parent_id
                    MERGE (ca)-[:MAPS_TO_TECHNIQUE]->(t)
                    RETURN count(*) AS cnt
                """, capec_id=capec_id, tech_id=tech_id, parent_id=parent_id)
                atk_links += result.single()['cnt']
        print(f"  → {atk_links} MAPS_TO_TECHNIQUE relationships")

    print("\n✅ CAPEC ingestion complete!")

ingest_capec_full(CAPEC_FILE)

Found 615 CAPEC patterns
Creating CAPEC nodes…
Linking CAPEC → CWE…
  → 1214 TARGETS_WEAKNESS relationships
Building CAPEC hierarchy…
  → 533 CHILD_OF relationships
Linking CAPEC → ATT&CK Techniques…
  → 432 MAPS_TO_TECHNIQUE relationships

✅ CAPEC ingestion complete!


---
## Cell 4 — Ingest CVEs from NVD
For each CVE: creates the CVE node, fetches its CWEs from NVD, and builds `CVE-[:CAUSE_WEAKNESS]->CWE`.

Supply your list of CVE IDs. Optionally add an NVD API key for higher rate limits (50 req/s vs 5 req/s).

In [6]:
import requests, nvdlib, time
from neo4j import GraphDatabase

def build_cve_list():
    all_cves = set()

    # 1. CISA KEV — highest priority, all actively exploited
    print("=== Fetching CISA KEV ===")
    url = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"
    kev = requests.get(url).json()
    kev_ids = {v["cveID"] for v in kev["vulnerabilities"]}
    all_cves.update(kev_ids)
    print(f"  +{len(kev_ids)} from CISA KEV")

    # 2. NVD query per CWE already in graph
    print("\n=== Fetching from graph CWEs ===")
    driver = GraphDatabase.driver(URI, auth=AUTH)
    with driver.session() as s:
        cwe_ids = [r["cwe_id"] for r in s.run("MATCH (w:CWE) RETURN w.id AS cwe_id")]
    driver.close()

    for cwe_id in cwe_ids:
        try:
            results = nvdlib.searchCVE(cweId=cwe_id, limit=5)
            ids = {r.id for r in results}
            all_cves.update(ids)
            print(f"  {cwe_id} → +{len(ids)} CVEs")
            time.sleep(0.7)
        except Exception as e:
            print(f"  {cwe_id} → error: {e}")

    print(f"\n✅ Total unique CVEs to ingest: {len(all_cves)}")
    return list(all_cves)

CVE_IDS = build_cve_list()

=== Fetching CISA KEV ===
  +1590 from CISA KEV

=== Fetching from graph CWEs ===
  CWE-276 → +5 CVEs
  CWE-285 → +5 CVEs
  CWE-434 → +5 CVEs
  CWE-693 → +5 CVEs
  CWE-732 → +5 CVEs
  CWE-1191 → +1 CVEs
  CWE-1193 → +0 CVEs
  CWE-1220 → +5 CVEs
  CWE-1297 → +0 CVEs
  CWE-1311 → +0 CVEs
  CWE-1314 → +0 CVEs
  CWE-1315 → +0 CVEs
  CWE-1318 → +0 CVEs
  CWE-1320 → +0 CVEs
  CWE-1321 → +5 CVEs
  CWE-1327 → +0 CVEs
  CWE-120 → +5 CVEs
  CWE-302 → +4 CVEs
  CWE-118 → +5 CVEs
  CWE-119 → +5 CVEs
  CWE-74 → +5 CVEs
  CWE-99 → +5 CVEs
  CWE-20 → +5 CVEs
  CWE-680 → +1 CVEs
  CWE-733 → +0 CVEs
  CWE-697 → +5 CVEs
  CWE-131 → +5 CVEs
  CWE-129 → +5 CVEs
  CWE-805 → +5 CVEs
  CWE-97 → +1 CVEs
  CWE-294 → +5 CVEs
  CWE-522 → +5 CVEs
  CWE-523 → +3 CVEs
  CWE-319 → +5 CVEs
  CWE-614 → +3 CVEs
  CWE-1021 → +5 CVEs
  CWE-250 → +5 CVEs
  CWE-638 → +0 CVEs
  CWE-116 → +5 CVEs
  CWE-113 → +5 CVEs
  CWE-138 → +3 CVEs
  CWE-436 → +5 CVEs
  CWE-648 → +5 CVEs
  CWE-89 → +5 CVEs
  CWE-78 → +5 CVEs
  CWE-114 → 

In [7]:
import nvdlib
import time

# ── Config ────────────────────────────────────────────────────────────────────
NVD_API_KEY = None          # Optional: set your key for higher rate limits
DELAY = 0.7                 # seconds between NVD calls (no key: ≤5 req/s)

# ── NVD fetch helper ──────────────────────────────────────────────────────────
def fetch_cve(cve_id):
    """Return (cvss_score, cwe_list, nvd_notes) for a CVE ID."""
    kwargs = {"cveId": cve_id}
    if NVD_API_KEY:
        kwargs["key"] = NVD_API_KEY
    try:
        results = nvdlib.searchCVE(**kwargs)
    except Exception as e:
        print(f"  NVD error for {cve_id}: {e}")
        return None, [], []      # ← added third return value
    if not results:
        return None, [], []      # ← added third return value

    r = results[0]
    cwes = []
    nvd_notes = []
    if hasattr(r, 'weaknesses'):
        for w in r.weaknesses:
            for d in w.description:
                if d.value.startswith('CWE-'):
                    cwes.append(d.value)
                else:
                    nvd_notes.append(d.value)

    score = None
    try:
        score = r.score[1]
    except Exception:
        pass

    return score, cwes, nvd_notes   # ← now returns all three

# ── Ingestion ─────────────────────────────────────────────────────────────────
def ingest_cves(cve_ids):
    print(f"Ingesting {len(cve_ids)} CVEs…")
    total_links = 0

    with driver.session() as s:
        for cve_id in cve_ids:
            score, cwes, nvd_notes = fetch_cve(cve_id)   # ← unpack three values
            time.sleep(DELAY)

            s.run("""
                MERGE (v:CVE {id: $id})
                SET v.cvss_score = $score,
                    v.nvd_weakness_notes = $notes
            """, id=cve_id, score=score, notes=nvd_notes)

            for cwe in cwes:
                s.run("""
                    MATCH (v:CVE {id: $cve_id})
                    MERGE (w:CWE {id: $cwe_id})
                    MERGE (v)-[:CAUSE_WEAKNESS]->(w)
                """, cve_id=cve_id, cwe_id=cwe)
                total_links += 1
                print(f"  {cve_id} (score={score}) → {cwe}")

            if not cwes:
                print(f"  {cve_id} — no CWEs found (nvd_note: {nvd_notes})")

    print(f"\n✅ CVE ingestion complete! {total_links} CAUSE_WEAKNESS links created.")

ingest_cves(CVE_IDS)

Ingesting 2717 CVEs…
  CVE-2025-4428 (score=7.2) → CWE-94
  CVE-2020-17463 (score=9.8) → CWE-89
  CVE-2020-17463 (score=9.8) → CWE-89
  CVE-2017-5161 (score=7.2) → CWE-427
  CVE-2018-14634 (score=7.8) → CWE-190
  CVE-2018-14634 (score=7.8) → CWE-190
  CVE-2026-35643 (score=8.6) → CWE-940
  CVE-2021-1905 (score=8.4) → CWE-416
  CVE-2021-1905 (score=8.4) → CWE-416
  CVE-2021-36741 (score=8.8) → CWE-434
  CVE-2021-36741 (score=8.8) → CWE-434
  CVE-2017-13083 (score=5.3) → CWE-295
  CVE-2017-13083 (score=5.3) → CWE-345
  CVE-2017-13083 (score=5.3) → CWE-347
  CVE-2017-13083 (score=5.3) → CWE-494
  CVE-2017-13083 (score=5.3) → CWE-494
  CVE-2012-5571 (score=5.4) → CWE-639
  CVE-2012-5571 (score=5.4) → CWE-255
  CVE-2026-21525 (score=6.2) → CWE-476
  CVE-2003-0721 (score=7.5) → CWE-129
  CVE-1999-0994 (score=5.0) → CWE-255
  CVE-2016-7462 (score=8.5) → CWE-264
  CVE-2016-7462 (score=8.5) → CWE-749
  CVE-2019-5825 (score=6.5) → CWE-787
  CVE-2019-5825 (score=6.5) → CWE-787
  CVE-2019-2725 (sc

---
## Cell 5 — Graph health check

In [8]:
CHECKS = [
    # Node counts
    ("Tactics",      "MATCH (n:Tactic)    RETURN count(n) AS cnt"),
    ("Techniques",   "MATCH (n:Technique) WHERE NOT n.is_subtechnique RETURN count(n) AS cnt"),
    ("Sub-techniques","MATCH (n:Technique) WHERE n.is_subtechnique RETURN count(n) AS cnt"),
    ("Groups",       "MATCH (n:Group)     RETURN count(n) AS cnt"),
    ("Software",     "MATCH (n:Software)  RETURN count(n) AS cnt"),
    ("CAPEC",        "MATCH (n:CAPEC)     RETURN count(n) AS cnt"),
    ("CWE",          "MATCH (n:CWE)       RETURN count(n) AS cnt"),
    ("CVE",          "MATCH (n:CVE)       RETURN count(n) AS cnt"),
    # Relationship counts
    ("BELONGS_TO (Technique→Tactic)",   "MATCH ()-[r:BELONGS_TO]->() RETURN count(r) AS cnt"),
    ("HAS_SUBTECHNIQUE",                 "MATCH ()-[r:HAS_SUBTECHNIQUE]->() RETURN count(r) AS cnt"),
    ("USES (Group/SW → Technique)",      "MATCH ()-[r:USES]->() RETURN count(r) AS cnt"),
    ("MAPS_TO_TECHNIQUE (CAPEC→ATT&CK)","MATCH ()-[r:MAPS_TO_TECHNIQUE]->() RETURN count(r) AS cnt"),
    ("TARGETS_WEAKNESS (CAPEC→CWE)",     "MATCH ()-[r:TARGETS_WEAKNESS]->() RETURN count(r) AS cnt"),
    ("CHILD_OF (CAPEC hierarchy)",       "MATCH ()-[r:CHILD_OF]->() RETURN count(r) AS cnt"),
    ("CAUSE_WEAKNESS (CVE→CWE)",         "MATCH ()-[r:CAUSE_WEAKNESS]->() RETURN count(r) AS cnt"),
    # Orphan checks
    ("Orphaned CVE  (no CWE)",  "MATCH (v:CVE) WHERE NOT (v)-[:CAUSE_WEAKNESS]->() RETURN count(v) AS cnt"),
    ("Orphaned CWE  (no edges)","MATCH (w:CWE) WHERE NOT ()-[]-(w) RETURN count(w) AS cnt"),
    ("Orphaned CAPEC (no ATT&CK link)","MATCH (ca:CAPEC) WHERE NOT (ca)-[:MAPS_TO_TECHNIQUE]->() RETURN count(ca) AS cnt"),
    ("Truly isolated CAPEC (no ATT&CK AND no CWE)", "MATCH (ca:CAPEC) WHERE NOT (ca)-[:MAPS_TO_TECHNIQUE]->() AND NOT (ca)-[:TARGETS_WEAKNESS]->() RETURN count(ca) AS cnt"),
]

print(f"{'Metric':<42} {'Count':>8}")
print("-" * 52)
with driver.session() as s:
    for label, query in CHECKS:
        result = s.run(query).single()
        cnt = result['cnt'] if result else 0
        flag = "  ⚠️" if 'Orphaned' in label and cnt > 0 else ""
        print(f"{label:<42} {cnt:>8}{flag}")

print("\n✅ Health check complete.")

Metric                                        Count
----------------------------------------------------
Tactics                                          28
Techniques                                      216
Sub-techniques                                  475
Groups                                          187
Software                                        784
CAPEC                                           615
CWE                                             416
CVE                                            2774
BELONGS_TO (Technique→Tactic)                   887
HAS_SUBTECHNIQUE                                477
USES (Group/SW → Technique)                   16102
MAPS_TO_TECHNIQUE (CAPEC→ATT&CK)                399
TARGETS_WEAKNESS (CAPEC→CWE)                   1262
CHILD_OF (CAPEC hierarchy)                      533
CAUSE_WEAKNESS (CVE→CWE)                       2831
Orphaned CVE  (no CWE)                          343  ⚠️
Orphaned CWE  (no edges)                          0
Orphane

---
## Cell 6 — Example queries (read-only, run anytime)

These are starter Cypher queries to explore your graph.

In [9]:
SAMPLE_QUERIES = {

    "Full chain: CVE → CWE → CAPEC → Technique → Tactic": """
        MATCH (v:CVE)-[:CAUSE_WEAKNESS]->(w:CWE)
              <-[:TARGETS_WEAKNESS]-(ca:CAPEC)
              -[:MAPS_TO_TECHNIQUE]->(t:Technique)
              -[:BELONGS_TO]->(ta:Tactic)
        RETURN v.id, w.id, ca.id, ca.name, t.mitre_id, t.name, ta.name
        LIMIT 10
    """,

    "Groups using a specific technique (e.g. T1059 — Command Scripting)": """
        MATCH (g:Group)-[:USES]->(t:Technique {mitre_id: 'T1059'})
        RETURN g.name, g.mitre_id
        ORDER BY g.name
    """,

    "Top 10 most-used techniques across all groups": """
        MATCH (g:Group)-[:USES]->(t:Technique)
        RETURN t.mitre_id, t.name, count(g) AS group_count
        ORDER BY group_count DESC
        LIMIT 10
    """,

    "Software used by a group (e.g. APT29)": """
        MATCH (g:Group {name: 'APT29'})-[:USES]->(s:Software)
        RETURN s.name, s.type, s.mitre_id
    """,

    "Techniques under Credential Access tactic": """
        MATCH (t:Technique)-[:BELONGS_TO]->(ta:Tactic {short_name: 'credential-access'})
        WHERE NOT t.is_subtechnique
        RETURN t.mitre_id, t.name
        ORDER BY t.mitre_id
    """,

    "CAPEC→CWE→CVE for a given technique": """
        MATCH (ca:CAPEC)-[:MAPS_TO_TECHNIQUE]->(t:Technique {mitre_id: 'T1134'})
        OPTIONAL MATCH (ca)-[:TARGETS_WEAKNESS]->(w:CWE)
        OPTIONAL MATCH (v:CVE)-[:CAUSE_WEAKNESS]->(w)
        RETURN t.name, ca.id, ca.name, w.id, v.id
    """,
}

with driver.session() as s:
    for title, query in SAMPLE_QUERIES.items():
        print(f"\n{'='*60}")
        print(f"  {title}")
        print('='*60)
        rows = s.run(query).data()
        if rows:
            for row in rows:
                print(row)
        else:
            print("  (no results)")


  Full chain: CVE → CWE → CAPEC → Technique → Tactic
{'v.id': 'CVE-2023-2897', 'w.id': 'CWE-345', 'ca.id': 'CAPEC-141', 'ca.name': 'Cache Poisoning', 't.mitre_id': 'T1557', 't.name': 'Adversary-in-the-Middle', 'ta.name': 'Credential Access'}
{'v.id': 'CVE-2014-0364', 'w.id': 'CWE-345', 'ca.id': 'CAPEC-141', 'ca.name': 'Cache Poisoning', 't.mitre_id': 'T1557', 't.name': 'Adversary-in-the-Middle', 'ta.name': 'Credential Access'}
{'v.id': 'CVE-2017-9606', 'w.id': 'CWE-345', 'ca.id': 'CAPEC-141', 'ca.name': 'Cache Poisoning', 't.mitre_id': 'T1557', 't.name': 'Adversary-in-the-Middle', 'ta.name': 'Credential Access'}
{'v.id': 'CVE-2014-8165', 'w.id': 'CWE-345', 'ca.id': 'CAPEC-141', 'ca.name': 'Cache Poisoning', 't.mitre_id': 'T1557', 't.name': 'Adversary-in-the-Middle', 'ta.name': 'Credential Access'}
{'v.id': 'CVE-2014-4936', 'w.id': 'CWE-345', 'ca.id': 'CAPEC-141', 'ca.name': 'Cache Poisoning', 't.mitre_id': 'T1557', 't.name': 'Adversary-in-the-Middle', 'ta.name': 'Credential Access'}
{

In [10]:
MIGRATIONS = [
    # 1. Rename :Weakness → :CWE
    (
        "Promote :Weakness to :CWE",
        """
        MATCH (w:Weakness)
        MERGE  (c:CWE {id: w.id})
        SET    c += properties(w)
        WITH   w, c
        MATCH  (w)-[r]->(nb)
        MERGE  (c)-[:TARGETS_WEAKNESS]->(nb)
        DETACH DELETE w
        """
    ),
    # 2. EXPLOITED_BY → CAUSE_WEAKNESS
    (
        "Replace :EXPLOITED_BY with :CAUSE_WEAKNESS",
        """
        MATCH (v:CVE)-[r:EXPLOITED_BY]->(w:CWE)
        MERGE (v)-[:CAUSE_WEAKNESS]->(w)
        DELETE r
        """
    ),
    # 3. Remove catch-all :RELATED edges (replaced by typed relationships)
    (
        "Delete generic :RELATED relationships",
        "MATCH ()-[r:RELATED]->() DELETE r"
    ),
]

print("Running v1 → v2 migrations…")
with driver.session() as s:
    for label, cypher in MIGRATIONS:
        print(f"  {label}…")
        s.run(cypher)
print("\n✅ Migration complete!")

Running v1 → v2 migrations…
  Promote :Weakness to :CWE…
  Replace :EXPLOITED_BY with :CAUSE_WEAKNESS…
  Delete generic :RELATED relationships…

✅ Migration complete!
